<a href="https://colab.research.google.com/github/YaS16s/Deep-learning/blob/main/Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
runtime under Runtime -> Change runtime type -> T4 GPU")# Install required libraries
!pip install ultralytics lap opencv-python-headless --quiet

import torch
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO

print(f"CUDA Hardware Check: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Please enable GPU

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 496.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA Hardware Check: False


In [12]:
yolo11_p2_yaml = """
nc: 10  # VisDrone target categories

# Backbone (feature extraction)
backbone:
  - [-1, 1, Conv, [64, 3, 2]]      # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]     # 1-P2/4 (Shallow spatial layer preserved)
  - [-1, 1, C3, [256, 1, True]] # 2 (c2, n_bottlenecks, shortcut)
  - [-1, 1, Conv, [256, 3, 2]]     # 3-P3/8
  - [-1, 1, C3, [512, 1, True]] # 4 (c2, n_bottlenecks, shortcut)
  - [-1, 1, Conv, [512, 3, 2]]     # 5-P4/16
  - [-1, 1, C3, [512, 1, True]]     # 6 (c2, n_bottlenecks, shortcut)
  - [-1, 1, Conv, [1024, 3, 2]]    # 7-P5/32
  - [-1, 1, C3, [1024, 1, True]]    # 8 (c2, n_bottlenecks, shortcut)
  - [-1, 1, SPPF, [1024, 5]]       # 9
  - [-1, 1, C2, [1024, 1, False]]         # 10 (c2, n_bottlenecks, shortcut)

# Neck (feature fusion with high-resolution P2 path - Temporarily removed for debugging)
neck:
  # - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  # - [[-1, 6], 1, Concat, [1]]
  # - [-1, 1, C3, [512, 1, True]]
  #
  # - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  # - [[-1, 4], 1, Concat, [1]]
  # - [-1, 1, C3, [256, 1, True]]
  #
  # - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  # - [[-1, 2], 1, Concat, [1]]
  # - [-1, 1, C3, [128, 1, True]]
  #
  # - [-1, 1, Conv, [128, 3, 2]]
  # - [[-1, 16], 1, Concat, [1]]
  # - [-1, 1, C3, [256, 1, True]]
  #
  # - [-1, 1, Conv, [256, 3, 2]]
  # - [[-1, 13], 1, Concat, [1]]
  # - [-1, 1, C3, [512, 1, True]]
  #
  # - [-1, 1, Conv, [512, 3, 2]]
  # - [[-1, 10], 1, Concat, [1]]
  # - [-1, 1, C3, [1024, 1, True]]

# Detect head receiving the last backbone output
head:
  - [[10], 1, Detect, [nc]] # Simplified to connect directly to the last backbone layer
"""

with open("yolo11-p2.yaml", "w") as f:
    f.write(yolo11_p2_yaml)

print("yolo11-p2.yaml successfully generated.")

yolo11-p2.yaml successfully generated.


In [5]:
# Instantiate model from custom structure definition
model = YOLO("yolo11-p2.yaml")

# Test forward pass with a dummy tensor to ensure no dimension mismatches
dummy_input = torch.randn(1, 3, 640, 640)
print(f"Custom Architecture Layers Built: {len(model.model.yaml['backbone']) + len(model.model.yaml['neck']) + 1}")
print("Network architecture compilation verified!")

WARNING ⚠️ no model scale passed. Assuming scale='n'.


IndexError: list index out of range

In [ ]:
# Train your custom P2 model on the VisDrone aerial benchmark
# The dataset (~2GB) auto-downloads on the first execution
results = model.train(
    data="VisDrone.yaml",
    epochs=1,           # Change to 50 or 100 for your actual research paper training
    imgsz=1280,         # High resolution preserves tiny high-altitude objects
    batch=4,            # Optimized for Colab free T4 VRAM limits
    device=0,
    save=True
)

print("Training cycle verified. Weights saved to runs/detect/train/weights/best.pt")

In [ ]:
# Fetch a sample video file
!wget -O sample_traffic.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/free-way-traffic.mp4

# Run state-of-the-art Deep OC-SORT multi-object tracking loop
# Using your customized small-object detector weights
model = YOLO("runs/detect/train/weights/best.pt")

tracking_results = model.track(
    source="sample_traffic.mp4",
    tracker="deepocsort.yaml", # Uses observation-centric spatial tracking
    conf=0.35,
    imgsz=1280,
    save=True,
    device=0
)

print("Tracking complete! Output video rendered to runs/detect/track/")

In [ ]:
# Latency & FPS Benchmarking Engine
cap = cv2.VideoCapture("sample_traffic.mp4")
frame_latencies = []
frame_counter = 0

print("Benchmarking performance on CUDA device...")

while frame_counter < 100:
    ret, frame = cap.read()
    if not ret:
        break

    t_start = time.time()
    _ = model(frame, imgsz=1280, verbose=False)
    t_end = time.time()

    frame_latencies.append((t_end - t_start) * 1000) # Convert to ms
    frame_counter += 1

cap.release()

avg_latency = np.mean(frame_latencies)
calculated_fps = 1000 / avg_latency

print("\n" + "="*50)
print("       EXPERIMENTAL RESEARCH BENCHMARKS      ")
print("="*50)
print(f"Average Frame Latency: {avg_latency:.2f} ms")
print(f"Real-time FPS Output : {calculated_fps:.2f} FPS")
if calculated_fps >= 25:
    print("Deployment Feasibility: MET (Real-Time Capable >= 25 FPS)")
else:
    print("Deployment Feasibility: Edge FP16 Export Optimization Recommended")
print("="*50)